In [1]:
import time
import numpy as np
from matplotlib import pyplot as plt
from scipy.interpolate import interp1d

from fisherA2Z.fisher_flex import FisherFlex
from fisherA2Z import nz_decomposition as nzd

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

In [2]:
ALPHA_DIR = "fisher_flex_data/hscy1_alpha_sampling/alphas/"
ALPHA_FILES = [
    "result_photometry_0_3_0_6_alpha.dat",
    "result_photometry_0_6_0_9_alpha.dat",
    "alpha_result_0_9_1_2.dat",
    "alpha_result_1_2_1_5.dat",
]
BIN_LABELS = ["0.3 < z < 0.6", "0.6 < z < 0.9", "0.9 < z < 1.2", "1.2 < z < 1.5"]

N_REAL = 200                                 # realizations used to set the priors
z_grid = np.linspace(0.02, 3.0, 150)         # common grid; the alpha files stop at z < 2

rng = np.random.default_rng(42)
per_bin = []
for fname in ALPHA_FILES:
    alpha = np.loadtxt(ALPHA_DIR + fname)     # columns: z, alpha
    draws = rng.dirichlet(alpha[:, 1], size=N_REAL)
    draws /= np.trapz(draws, alpha[:, 0], axis=1)[:, None]
    # Each bin has its own z sampling, so interpolate onto the shared grid.
    on_grid = interp1d(alpha[:, 0], draws, kind="linear",
                       bounds_error=False, fill_value=0.0)(z_grid)
    on_grid = np.clip(on_grid, 0.0, None)
    per_bin.append(on_grid / np.trapz(on_grid, z_grid, axis=1)[:, None])

# FisherFlex wants (n_realizations, n_tomo, n_z).
nz_realizations = np.swapaxes(np.array(per_bin), 0, 1)
nz_source = nz_realizations.mean(axis=0)
nz_source /= np.trapz(nz_source, z_grid, axis=1)[:, None]

print("realizations:", nz_realizations.shape)
print("central n(z):", nz_source.shape)
print("mean redshift per bin:",
      np.round(np.trapz(nz_source * z_grid, z_grid, axis=1), 3))

realizations: (200, 4, 150)
central n(z): (4, 150)
mean redshift per bin: [0.464 0.745 1.048 1.325]


In [26]:
t0 = time.time()
flex_y1_cs = FisherFlex(
    # -- the n(z) and its uncertainty ---------------------------------
    nz_source=nz_source,
    nz_realizations=nz_realizations,
    z_grid=z_grid,
    # -- the survey ---------------------------------------------------
    neff_source=[4, 5, 4, 2],   # arcmin^-2, per tomographic bin
    fsky=0.0364,                        # ~150 deg^2, HSC Y1
    sigma_e=0.26,                       # per-component ellipticity dispersion
    # -- the analysis -------------------------------------------------
    mode="cosmic_shear",                       # or '2x2pt' / 'cosmic_shear'
    nz_model="shift_stretch",           # the default; see below for the alternative
)
print(f"\n__init__ took {time.time() - t0:.1f} s")




Measuring the shift and stretch of 200 realizations ...

__init__ took 0.1 s


In [29]:
flex_y1_cs.compute(parallel=True)

res_y1_cs = flex_y1_cs.forecast(ell_max_cs=1800, ell_min_cs=300)

Computing the fiducial data vector and covariance ...
Computing derivatives for 29 parameters (parallel) ...
Done.


In [19]:
flex_y3_cs = FisherFlex(
    # -- the n(z) and its uncertainty ---------------------------------
    nz_source=nz_source,
    nz_realizations=nz_realizations,
    z_grid=z_grid,
    # -- the survey ---------------------------------------------------
    neff_source=[4, 5, 4, 2],   # arcmin^-2, per tomographic bin
    fsky=0.0364*3,                        # ~150 deg^2, HSC Y1
    sigma_e=0.26,                       # per-component ellipticity dispersion
    # -- the analysis -------------------------------------------------
    mode="cosmic_shear",                       # or '2x2pt' / 'cosmic_shear'
    nz_model="shift_stretch",           # the default; see below for the alternative
)

Measuring the shift and stretch of 200 realizations ...


In [23]:
flex_y3_cs.compute(parallel=True)

res_y3_cs = flex_y3_cs.forecast(ell_max_cs=1800, ell_min_cs=300)

Computing the fiducial data vector and covariance ...
Computing derivatives for 29 parameters (parallel) ...
Done.


In [32]:
s8, s8_err = res_y1_cs.s8()
print(f"S_8_HSCY1 = {s8:.4f} +/- {s8_err:.4f}")

S_8_HSCY1 = 0.8523 +/- 0.0257


In [33]:
s8, s8_err = res_y3_cs.s8()
print(f"S_8_HSCY3 = {s8:.4f} +/- {s8_err:.4f}")

S_8_HSCY3 = 0.8523 +/- 0.0190
